In [ ]:
import sys
import pathlib

sys.path.append(str(pathlib.Path('..').resolve()))

%load_ext autoreload
%autoreload 2

from evaluation.utils import get_summary_df, load_model
from dataset.dataset import SetKnowledgeTrendingSinusoidsDistShift
from dataset.utils import get_dataloader

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

sns.set_style('whitegrid')
sns.set_palette('Dark2')
plt.rcParams["text.latex.preamble"] = (
    "\usepackage{lmodern} \usepackage{times} \usepackage{amssymb}"
)


In [ ]:

# Load the models
save_dirs = {
    'NP': '../saves/INPs_sinusoids/np_dist_shift_0',
    'INP': '../saves/INPs_sinusoids/inp_b_dist_shift_0',
}

model_dict = {}
config_dict = {}
for model_name, save_dir in save_dirs.items():
    model, cfg = load_model(save_dir, load_it='best')
    model.eval()
    model_dict[model_name] = model
    config_dict[model_name] = cfg

model_names = list(model_dict.keys())


In [ ]:
from argparse import Namespace

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
config = Namespace(
    min_num_context=0,
    max_num_context=100,
    num_targets=100,
    noise=0.2,
    batch_size=25,
    x_sampler='uniform',
    test_num_z_samples=32,
    dataset='set-trending-sinusoids-dist-shift',
    device=device,
)

train_dataset = SetKnowledgeTrendingSinusoidsDistShift(
    root='../data/trending-sinusoids-dist-shift', split='train', knowledge_type='b'
)
train_data_loader = get_dataloader(train_dataset, config)

test_dataset = SetKnowledgeTrendingSinusoidsDistShift(
    root='../data/trending-sinusoids-dist-shift', split='test', knowledge_type='b'
)
test_data_loader = get_dataloader(test_dataset, config)


In [ ]:

# Evaluate baseline, shuffled knowledge, and FK variants
eval_type_ls = ['raw', 'informed', 'shuffled', 'random']

train_summary_df, train_losses, train_output_dict = get_summary_df(
    model_dict, config_dict, train_data_loader, eval_type_ls, model_names
)
test_summary_df, test_losses, test_output_dict = get_summary_df(
    model_dict, config_dict, test_data_loader, eval_type_ls, model_names
)

train_summary_df['split'] = 'train'
test_summary_df['split'] = 'test'


In [ ]:

import re

def _select_best(df, model_name, base_eval, kind='fk_lambda'):
    if kind == 'fk_lambda':
        pattern = rf"^{re.escape(base_eval)}_fk_lambda_"
        value_col = 'lambda'
        extractor = r"lambda_([0-9.]+)"
    else:
        pattern = rf"^{re.escape(base_eval)}_fkdr_alpha_"
        value_col = 'alpha'
        extractor = r"alpha_([0-9.]+)"

    sub = df[(df.model_name == model_name) & (df.eval_type.str.match(pattern))].copy()
    if sub.empty:
        return sub
    sub[value_col] = sub.eval_type.str.extract(extractor).astype(float)
    best = (
        sub.sort_values(['num_context', 'mean'])
        .groupby('num_context', as_index=False)
        .first()
    )
    best['eval_type'] = f"{base_eval}_{kind}_best"
    best['best_hyper'] = best[value_col]
    return best


In [ ]:
# Build plot data from train and test splits (average log-likelihood)
base_labels = {
    'raw': r"NP: $\mathcal{K} = \varnothing$",
    'informed': r"INP: $\mathcal{K} \neq \varnothing$",
}

combined_df = pd.concat([train_summary_df, test_summary_df], ignore_index=True, sort=False)
base_df = combined_df[
    (combined_df.eval_type.isin(base_labels)) & (combined_df.model_name.isin(['NP', 'INP']))
].copy()
base_df['eval_label'] = base_df['eval_type'].map(base_labels)
base_df['avg_log_like'] = -base_df['mean']

fig, ax = plt.subplots(figsize=(4.2, 3.2))
sns.lineplot(
    base_df,
    x='num_context',
    y='avg_log_like',
    hue='eval_label',
    style='split',
    style_order=['train', 'test'],
    markers=True,
    ax=ax,
)
ax.set_ylabel('Average log-likelihood')
ax.set_xlabel('Number of context datapoints')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, title='')
plt.tight_layout()
plt.show()

# Build FK comparison on the test split (average log-likelihood)
base_labels_fk = {
    'raw': r"NP: $\mathcal{K} = \varnothing$",
    'informed': r"INP: $\mathcal{K} \neq \varnothing$",
    'shuffled': 'INP: K shuffled',
    'random': 'INP: K random',
}

test_df = test_summary_df.copy().dropna(subset=['mean'])
base_df_test = test_df[(test_df.eval_type.isin(base_labels_fk)) & (test_df.model_name.isin(['NP','INP']))].copy()
base_df_test['eval_label'] = base_df_test['eval_type'].map(base_labels_fk)
base_df_test['avg_log_like'] = -base_df_test['mean']

best_rows = []
for model_name, base_eval, kind, label_prefix in [
    ('NP', 'raw', 'fk_lambda', 'NP FK λ'),
    ('INP', 'informed', 'fk_lambda', 'INP FK λ'),
    ('INP', 'informed', 'fkdr_alpha', 'INP FK-DR α'),
    ('INP', 'shuffled', 'fk_lambda', 'INP shuffled FK λ'),
]:
    best = _select_best(test_df, model_name=model_name, base_eval=base_eval, kind=kind)
    if not best.empty:
        best['eval_label'] = best['best_hyper'].apply(lambda v: f"{label_prefix}={v:g}")
        best_rows.append(best)

if best_rows:
    fk_best_df = pd.concat(best_rows, ignore_index=True)
    fk_best_df['avg_log_like'] = -fk_best_df['mean']
else:
    fk_best_df = pd.DataFrame()

plot_df = pd.concat([base_df_test, fk_best_df], ignore_index=True, sort=False)

fig, ax = plt.subplots(figsize=(4.4, 3.2))
sns.lineplot(
    plot_df,
    x='num_context',
    y='avg_log_like',
    hue='eval_label',
    style='model_name',
    style_order=['NP', 'INP'],
    markers=True,
    ax=ax,
)
ax.set_ylabel('Average log-likelihood (test)')
ax.set_xlabel('Number of context datapoints')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, title='')
plt.tight_layout()
plt.show()


In [ ]:

# Summaries of best FK hyperparameters per context
best_tables = []
for label, kind in [('FK (curve fit)', 'fk_lambda'), ('FK density ratio', 'fkdr_alpha')]:
    for model_name, base_eval in [('NP', 'raw'), ('INP', 'informed'), ('INP', 'shuffled')]:
        best = _select_best(test_summary_df, model_name=model_name, base_eval=base_eval, kind=kind)
        if best.empty:
            continue
        tbl = best[['num_context', 'best_hyper']].rename(columns={'best_hyper': f'{label} best'}).copy()
        tbl['model'] = model_name
        tbl['eval_type'] = base_eval
        best_tables.append(tbl)

if best_tables:
    best_hyper_df = pd.concat(best_tables, ignore_index=True)
    display(
        best_hyper_df.pivot_table(
            index=['model', 'eval_type'],
            columns='num_context',
            values=[c for c in best_hyper_df.columns if c.endswith('best')],
        )
    )
else:
    print('No FK rows available to summarize.')
